# K-means in Python
200 students (CGPA, IQ): elbow method, k-means with k = 4, the clusters; then k-means on 3-D data.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

# Load the students: two columns, cgpa and iq
df = pd.read_csv("data/student_clustering.csv")
print("The shape of data is", df.shape)
df.head()

The shape of data is (200, 2)


,cgpa,iq
0,5.13,88
1,5.90,113
2,8.36,93
3,8.27,97
4,5.45,110


In [2]:
# Look at the data first
px.scatter(df, x="cgpa", y="iq", template="simple_white")

## Elbow method

In [3]:
# One k-means model per k; inertia_ is the WCSS of the fitted model
wcss = []
for i in range(1, 11):
    km = KMeans(n_clusters=i, random_state=0)
    km.fit(df)
    wcss.append(km.inertia_)
np.round(wcss, 1)

array([29957.9,  4184.1,  2503.4,   682. ,   530.4,   420.7,   385.4,
         294.1,   204.4,   181.2])

In [4]:
# The elbow curve: the bend is at k = 4
go.Figure(go.Scatter(x=list(range(1, 11)), y=wcss, mode="lines+markers")).update_layout(
    template="simple_white", xaxis_title="k", yaxis_title="WCSS (inertia_)")

## k-means with k = 4

In [5]:
# A NumPy array is easier to slice than a DataFrame
X = df.iloc[:, :].values
km = KMeans(n_clusters=4, random_state=0)
y_means = km.fit_predict(X)     # train, then return the cluster of every row
print(y_means[:10])
print("rows per cluster:", np.bincount(y_means))
print("centroids (cgpa, iq):\n", km.cluster_centers_.round(2))

[3 2 1 1 2 2 1 0 2 1]
rows per cluster: [50 50 50 50]
centroids (cgpa, iq):
 [[  8.87 117.16]
 [  8.2   94.6 ]
 [  5.89 109.52]
 [  4.97  86.7 ]]


In [6]:
# Boolean indexing: X[y_means == 0, 0] is the CGPA of every student in cluster 0
X[y_means == 3, 1][:5]

array([88., 86., 88., 86., 87.])

In [7]:
# One scatter trace per cluster
fig = go.Figure()
for c, colour in zip(range(4), ["blue", "red", "green", "gold"]):
    fig.add_trace(go.Scatter(x=X[y_means == c, 0], y=X[y_means == c, 1], mode="markers",
                             marker=dict(color=colour), name=f"cluster {c}"))
fig.update_layout(template="simple_white", xaxis_title="cgpa", yaxis_title="iq")

In [8]:
# Does scaling change the clusters here? Compare the two labelings (1.0 = identical groups)
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler
y_scaled = KMeans(n_clusters=4, random_state=0).fit_predict(StandardScaler().fit_transform(X))
print("agreement with the unscaled clusters:", adjusted_rand_score(y_means, y_scaled))

agreement with the unscaled clusters: 1.0


In [9]:
# Why no change? Check whether the four groups are a fixed point of k-means in both spaces:
# every student nearer its own centroid than any other centroid (ratio > 1)
from sklearn.preprocessing import StandardScaler
for name, Z in [("raw", X), ("standardized", StandardScaler().fit_transform(X))]:
    C = np.array([Z[y_means == k].mean(0) for k in range(4)])
    D = np.linalg.norm(Z[:, None] - C[None], axis=2)
    own = D[np.arange(len(Z)), y_means].copy()
    D[np.arange(len(Z)), y_means] = np.inf
    print(name, "smallest (nearest other centroid / own centroid):", round((D.min(1) / own).min(), 2))

raw smallest (nearest other centroid / own centroid): 1.08
standardized smallest (nearest other centroid / own centroid): 2.56


## k-means on 3-D data

In [10]:
# 200 points around 4 centres in 3 dimensions
centroids = [(-5, -5, 5), (5, 5, -5), (3.5, -2.5, 4), (-2.5, 2.5, -4)]
X3, _ = make_blobs(n_samples=200, centers=centroids, cluster_std=1, n_features=3, random_state=1)
px.scatter_3d(x=X3[:, 0], y=X3[:, 1], z=X3[:, 2])

In [11]:
# Exactly the same elbow code
wcss3 = [KMeans(n_clusters=i, random_state=0).fit(X3).inertia_ for i in range(1, 21)]
go.Figure(go.Scatter(x=list(range(1, 21)), y=wcss3, mode="lines+markers")).update_layout(
    template="simple_white", xaxis_title="k", yaxis_title="WCSS (inertia_)")

In [12]:
# Train with k = 4 and colour the points by cluster
y_pred = KMeans(n_clusters=4, random_state=0).fit_predict(X3)
d3 = pd.DataFrame({"col1": X3[:, 0], "col2": X3[:, 1], "col3": X3[:, 2], "label": y_pred.astype(str)})
px.scatter_3d(d3, x="col1", y="col2", z="col3", color="label")